In [1]:
import warnings
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor

warnings.filterwarnings("ignore")

ROOT = r"C:\Users\reinh\Downloads\Projects\Rossmann"
RAW_TRAIN = f"{ROOT}/Rossmann/train.csv"
RAW_STORE = f"{ROOT}/Rossmann/store.csv"
PROC = f"{ROOT}/rsmn"

HORIZON = 28
WINDOW_OFFSETS = [0, 28, 56]
ORIGINAL_STORES = [98, 480, 486, 735, 865]

train = pd.read_csv(RAW_TRAIN, dtype={"StateHoliday": str}, parse_dates=["Date"])
store = pd.read_csv(RAW_STORE)
calendar = pd.date_range(train["Date"].min(), train["Date"].max(), freq="D")

def max_gap(dates):
    missing = calendar.difference(pd.DatetimeIndex(dates))
    return 0 if len(missing) == 0 else int((pd.Series(missing).diff().dt.days != 1).cumsum().value_counts().max())

coverage = train.groupby("Store")["Date"].nunique() / len(calendar)
gaps = train.groupby("Store")["Date"].apply(max_gap)
eligible = coverage[(coverage >= 0.95) & (gaps < 14)].index.values

pool = [s for s in eligible if s not in ORIGINAL_STORES]
NEW_STORES = sorted(np.random.default_rng(2026).choice(pool, size=20, replace=False).tolist())
ALL_TEST = sorted(ORIGINAL_STORES + NEW_STORES)
print("Total test stores:", len(ALL_TEST))

Total test stores: 25


In [2]:
panel = (
    train.set_index(["Store", "Date"])
    .reindex(pd.MultiIndex.from_product([eligible, calendar], names=["Store", "Date"]))
    .reset_index()
    .merge(store, on="Store", how="left")
    .sort_values(["Store", "Date"])
    .reset_index(drop=True)
)

g = panel.groupby("Store")["Sales"]
for k in [28, 35, 42, 49]:
    panel[f"lag_{k}"] = g.shift(k)

panel["same_dow_mean_4w"] = panel[["lag_28", "lag_35", "lag_42", "lag_49"]].mean(axis=1)
panel["roll_mean_28"] = g.transform(lambda s: s.shift(HORIZON).rolling(HORIZON, min_periods=14).mean())

panel["DayOfWeek"] = panel["Date"].dt.dayofweek + 1
panel["month"] = panel["Date"].dt.month
panel["day_of_month"] = panel["Date"].dt.day
panel["week_of_year"] = panel["Date"].dt.isocalendar().week.astype(int)
panel["StateHoliday"] = panel["StateHoliday"].fillna("0").map({"0": 0, "a": 1, "b": 2, "c": 3})
panel["StoreType"] = panel["StoreType"].astype("category").cat.codes
panel["Assortment"] = panel["Assortment"].astype("category").cat.codes
panel["CompetitorMissing"] = panel["CompetitionDistance"].isna().astype(int)
panel["CompetitionDistance"] = panel["CompetitionDistance"].fillna(100_000)

FEATURES = [
    "DayOfWeek", "Promo", "StateHoliday", "SchoolHoliday",
    "month", "day_of_month", "week_of_year",
    "StoreType", "Assortment", "CompetitionDistance", "CompetitorMissing", "Promo2",
    "lag_28", "lag_35", "lag_42", "lag_49", "same_dow_mean_4w", "roll_mean_28",
]

In [3]:
def mape(actual, pred):
    return np.mean(np.abs(actual - pred) / actual) * 100

def wape(actual, pred):
    return np.sum(np.abs(actual - pred)) / np.sum(actual) * 100

rows = []
for off in WINDOW_OFFSETS:
    cut = calendar[-HORIZON - 1 - off]
    end = cut + pd.Timedelta(days=HORIZON)

    is_tr = (panel["Date"] <= cut) & (panel["Open"] == 1) & panel["Sales"].notna()
    model = HistGradientBoostingRegressor(max_iter=400, learning_rate=0.05, min_samples_leaf=100, early_stopping=False, random_state=42)
    model.fit(panel.loc[is_tr, FEATURES], np.log1p(panel.loc[is_tr, "Sales"]))

    ho = panel[(panel["Date"] > cut) & (panel["Date"] <= end) & panel["Store"].isin(ALL_TEST)].copy()
    ho["pred_gb"] = np.expm1(model.predict(ho[FEATURES])).clip(min=0)

    for s, d in ho.groupby("Store"):
        hist = panel[(panel["Store"] == s) & (panel["Date"] <= cut)].tail(HORIZON)
        by_dow = hist.groupby("DayOfWeek")["Sales"].mean()
        d = d[(d["Open"] == 1) & (d["Sales"] > 0)].copy()
        d["pred_naive"] = d["DayOfWeek"].map(by_dow)

        for name, col in [("Gradient boosting", "pred_gb"), ("Same-weekday mean", "pred_naive")]:
            rows.append({"cutoff": cut.date(), "Store": s, "Model": name, "MAPE_%": mape(d["Sales"], d[col]), "WAPE_%": wape(d["Sales"], d[col])})

res = pd.DataFrame(rows)
res["group"] = np.where(res["Store"].isin(ORIGINAL_STORES), "original 5", "new 20")

print("=== All 25 Stores Average ===")
print(res.groupby("Model")[["MAPE_%", "WAPE_%"]].mean().round(2))

print("\n=== New 20 Stores Only ===")
print(res[res["group"] == "new 20"].groupby("Model")[["MAPE_%", "WAPE_%"]].mean().round(2))

=== All 25 Stores Average ===
                   MAPE_%  WAPE_%
Model                            
Gradient boosting    9.03    8.84
Same-weekday mean   18.49   18.44

=== New 20 Stores Only ===
                   MAPE_%  WAPE_%
Model                            
Gradient boosting    8.99    8.80
Same-weekday mean   18.14   18.12


In [4]:
per_store = res.groupby(["Store", "Model"])["MAPE_%"].mean().unstack()
gap = per_store["Same-weekday mean"] - per_store["Gradient boosting"]

rng = np.random.default_rng(0)
boot_means = [rng.choice(gap.values, size=len(gap), replace=True).mean() for _ in range(5000)]
lo, hi = np.percentile(boot_means, [2.5, 97.5])

print(f"Average MAPE gain: {gap.mean():.2f} pp (95% CI: {lo:.2f} to {hi:.2f})")
print(f"Stores improved: {(gap > 0).sum()} of {len(gap)}")

res.round(2).to_csv(f"{PROC}/sample_extension_results.csv", index=False)
print("Saved sample_extension_results.csv")

Average MAPE gain: 9.46 pp (95% CI: 8.19 to 10.74)
Stores improved: 25 of 25
Saved sample_extension_results.csv
